# 06 - Advanced ML: Optuna, XGBoost, and Ensembles
Let's find the absolute best model using AI-driven hyperparameter tuning and a Voting Classifier.

In [1]:
!pip install xgboost optuna


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
import optuna
import joblib
import os

# 1. Load Data
df = pd.read_csv('../data/processed/ml_features.csv', index_col=0, parse_dates=True)
feature_columns = ['copper_Close', 'zinc_proxy_Close', 'usdinr_Close', 'crude_Close', 'copx_Close', 'Brass_Cost', 'Daily_Return', 'SMA_10', 'SMA_30', 'Volatility_14d']
X = df[feature_columns]
y = df['Target_Direction']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# ---------------------------------------------------------
# Phase 1: OPTUNA - Tuning XGBoost
# ---------------------------------------------------------
print(" PHASE 1: Optuna is hunting for the best XGBoost parameters...")

def objective(trial):
    param = {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }
    model = xgb.XGBClassifier(**param, random_state=42, eval_metric='logloss')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Hide massive logs
study.optimize(objective, n_trials=15)

print(f" Optuna finished! Best Accuracy found: {study.best_value * 100:.2f}%")
print(f"Best Parameters: {study.best_params}\n")

# ---------------------------------------------------------
# Phase 2: ENSEMBLE - The 'Wisdom of the Crowd'
# ---------------------------------------------------------
print("🤝 PHASE 2: Building the Ensemble (Voting) Classifier...")

tuned_xgb = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
lr_model = LogisticRegression(max_iter=2000, random_state=42)

ensemble_model = VotingClassifier(
    estimators=[('lr', lr_model), ('rf', rf_model), ('xgb', tuned_xgb)],
    voting='soft'
)

# ---------------------------------------------------------
# Phase 3: THE FINAL RACE
# ---------------------------------------------------------
print(" PHASE 3: The Final Model Showdown!\n" + "-"*40)

final_models = {
    "Baseline Random Forest": rf_model,
    "Optuna-Tuned XGBoost": tuned_xgb,
    "Ensemble (LR + RF + XGB)": ensemble_model
}

best_model = None
best_accuracy = 0
best_name = ""

for name, model in final_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f" {name:25} Accuracy: {acc * 100:.2f}%")
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_name = name

print("-" * 40)
print(f" ULTIMATE WINNER: {best_name} ({best_accuracy * 100:.2f}%)")

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/brass_ultimate_model.joblib')
print(f"\n Saved Ultimate Model to /models/brass_ultimate_model.joblib")

 PHASE 1: Optuna is hunting for the best XGBoost parameters...
 Optuna finished! Best Accuracy found: 47.25%
Best Parameters: {'max_depth': 3, 'learning_rate': 0.15956719921813523, 'n_estimators': 66, 'subsample': 0.650849196168171}

🤝 PHASE 2: Building the Ensemble (Voting) Classifier...
 PHASE 3: The Final Model Showdown!
----------------------------------------
 Baseline Random Forest    Accuracy: 46.52%
 Optuna-Tuned XGBoost      Accuracy: 47.25%
 Ensemble (LR + RF + XGB)  Accuracy: 46.64%
----------------------------------------
 ULTIMATE WINNER: Optuna-Tuned XGBoost (47.25%)

 Saved Ultimate Model to /models/brass_ultimate_model.joblib
